In [ ]:
# 1. Install Dependencies
!pip install -q streamlit langgraph langchain langchain-openai langchain-community pydantic pytest torch 'transformers>=4.51.0,<5' 'accelerate>=1.6.0' 'bitsandbytes>=0.46.0' 'pypdf>=5.4.0,<6'


In [ ]:
# 2. Clone a fresh copy of the project on every run
import getpass
import os
import shutil
from pathlib import Path

repo_name = "smart-wallet-optimizer"
repo_url = f"https://github.com/cloudbyte/{repo_name}.git"
current_dir = Path.cwd()
clone_parent = current_dir.parent if current_dir.name == repo_name else current_dir
repo_path = clone_parent / repo_name

if repo_path.exists():
    shutil.rmtree(repo_path)

print(f"Cloning a fresh copy of {repo_name}...")
clone_res = !git clone {repo_url} {repo_path} 2>&1
if any("fatal:" in line for line in clone_res):
    print("🔐 Private repository detected. Please enter your GitHub Personal Access Token (PAT):")
    token = getpass.getpass("GitHub PAT: ").strip()
    if token:
        auth_url = f"https://{token}@github.com/cloudbyte/{repo_name}.git"
        clone_res = !git clone {auth_url} {repo_path} 2>&1

if not repo_path.exists():
    raise RuntimeError("Git clone failed. Check the repository URL and your access permissions.")

os.chdir(repo_path)
print(f"Using fresh clone at {repo_path}")


In [ ]:
# 3. Run Tests
!PYTHONPATH=. pytest


In [ ]:
# 4. Start the Web App
import torch
import os
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen

if torch.cuda.is_available():
    os.environ["WALLET_AGENT_MODEL"] = "Qwen/Qwen3-4B"
    os.environ["WALLET_AGENT_DEVICE"] = "auto"
    os.environ["WALLET_AGENT_4BIT"] = "true"
    print(f"GPU detected ({torch.cuda.get_device_name(0)}); using Qwen3-4B.")
else:
    os.environ["WALLET_AGENT_MODEL"] = "Qwen/Qwen3-0.6B"
    os.environ["WALLET_AGENT_DEVICE"] = "cpu"
    os.environ["WALLET_AGENT_4BIT"] = "false"
    print("No GPU detected; using Qwen3-0.6B on CPU. For faster inference, select a GPU runtime.")

health_url = "http://127.0.0.1:8501/_stcore/health"
try:
    with urlopen(health_url, timeout=2) as response:
        app_is_running = response.status == 200
except Exception:
    app_is_running = False

if not app_is_running:
    log_file = open("/tmp/smart-wallet-streamlit.log", "w")
    streamlit_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", "src/web_app.py",
            "--server.port", "8501", "--server.address", "0.0.0.0",
            "--server.headless", "true",
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if streamlit_process.poll() is not None:
            raise RuntimeError(Path("/tmp/smart-wallet-streamlit.log").read_text())
        try:
            with urlopen(health_url, timeout=2) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Streamlit did not become ready. See /tmp/smart-wallet-streamlit.log")

from google.colab import output

print("Opening the app in a separate tab. Keep the Colab runtime connected.")
output.serve_kernel_port_as_window(8501)
